In [15]:
from pathlib import Path

# Project root (one level up from this notebook)
ROOT = Path("..").resolve()
RAW_DIR = ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {ROOT}")
print(f"Raw data dir: {RAW_DIR}")
print(f"Raw dir exists: {RAW_DIR.exists()}")


Project root: /Users/abnegusu/Desktop/Github/AI-Sec-Rag
Raw data dir: /Users/abnegusu/Desktop/Github/AI-Sec-Rag/data/raw
Raw dir exists: True


In [16]:
# See what's currently in our raw data folder
raw_files = sorted(RAW_DIR.iterdir())

print(f"Found {len(raw_files)} file(s) in {RAW_DIR.name}/")
for path in raw_files:
    print(f"  - {path.name}")


Found 2 file(s) in raw/
  - owasp-a01-broken-access-control.md
  - owasp-top-ten.html


## Step A — Raw inventory

Document what we ingested: each file in `data/raw/` with its size (bytes).


In [17]:
# Catalog: filename + size for audit trail
for path in sorted(RAW_DIR.iterdir()):
    if path.is_file():
        print(f"{path.name:45} {path.stat().st_size:,} bytes")


owasp-a01-broken-access-control.md            9,306 bytes
owasp-top-ten.html                            94,018 bytes


## Step B — Stage processed text

Copy our Markdown source into `data/processed/` as plain text for chunking/embeddings (Week 2+).


In [18]:
# Processed = downstream pipeline input (not re-downloaded each time)
PROCESSED_DIR = ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

source = RAW_DIR / "owasp-a01-broken-access-control.md"
text = source.read_text(encoding="utf-8")

dest = PROCESSED_DIR / "owasp-a01-broken-access-control.txt"
dest.write_text(text, encoding="utf-8")

print(f"Wrote {len(text):,} characters to {dest.relative_to(ROOT)}")


Wrote 9,304 characters to data/processed/owasp-a01-broken-access-control.txt


## Chunk (next)

**Input:** `data/processed/*.txt` from Step B.

**Goal:** Split long documents into smaller **chunks** (e.g. ~500 characters with overlap) so embeddings and retrieval work on focused pieces, not one huge file.

**Output (planned):** `data/chunks/` — one record per chunk (text + metadata: source file, chunk index).